Preprocessing Notebook for Labor Force History Data

In [ ]:
'''
This notebook is meant to preprocess the Labor Force History Data Set, details are listed below:

Source: https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=1410042101
Table: 14-10-0421-01
Geography: Canada and Province/Territory
Reference Period Range: 2000-01-01 to 2025-09-01
'''

In [2]:
import polars as pl 
import pandas as pd 

In [7]:
path = 'rawdata/1410042101_databaseLoadingData.csv'
dp_Raw_Data = pl.scan_csv(path)

In [8]:
#Understanding Schema and Structure
dp_Raw_Data.schema

C:\Users\tejas\AppData\Local\Temp\ipykernel_15840\958564254.py:2: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  dp_Raw_Data.schema


Schema([('REF_DATE', String),
        ('GEO', String),
        ('DGUID', String),
        ('Labour force characteristics', String),
        ('National Occupational Classification (NOC)', String),
        ('Gender', String),
        ('UOM', String),
        ('UOM_ID', Int64),
        ('SCALAR_FACTOR', String),
        ('SCALAR_ID', Int64),
        ('VECTOR', String),
        ('COORDINATE', String),
        ('VALUE', Float64),
        ('STATUS', String),
        ('SYMBOL', String),
        ('TERMINATED', String),
        ('DECIMALS', Int64)])

In [9]:
#Verifying Date ranges with pandas 
df_Unique_Dates = dp_Raw_Data.select('REF_DATE').unique().collect().to_pandas()
df_Unique_Dates['REF_DATE'] = pd.to_datetime(df_Unique_Dates['REF_DATE'])

In [10]:
#Checking Labor Force Characteristics 
df_Unique_Labor_Force_Characteristics = dp_Raw_Data.select('Labour force characteristics').unique().collect().to_pandas().dropna()
df_Unique_Labor_Force_Characteristics 

,Labour force characteristics
0,Labour force
1,Part-time employment
2,Full-time employment
4,Employment
5,Unemployment
6,Unemployment rate


In [ ]:
'''
Our interest is exlusively in 'Full-Time Employment' 
According to the meta data:

'Full-time employment consists of persons who usually work 30 hours 
or more per week at their main or only job. 
Estimates in thousands, rounded to the nearest hundred.'

This is the characteristic we will be trying to forecast, but we will retain the other characteristic. 


'''


In [24]:
df_Coordinate_Identifiers = dp_Raw_Data.select(['COORDINATE']).unique().collect().to_pandas()
df_Coordinate_Identifiers

,COORDINATE
0,1.2.38.1
1,4.4.32.1
2,11.6.42.1
3,3.2.32.1
4,10.6.36.1
...,...
3956,2.1.26.1
3957,10.4.34.1
3958,6.4.34.1
3959,2.5.55.1


In [41]:
#According to the meta data -> NOC Groupg Member ID is in the 3rd postion of the Coordinate
df_Coordinate_Identifiers['NOC_GroupingID'] = df_Coordinate_Identifiers['COORDINATE'].str.split(".").str[2]

#According to the meta data -> Gender membe ID is in the 4th position of the Coordinate
#but we downloaded a filtered set that only tracks the total for the estimate
df_Coordinate_Identifiers['Gender_MemberID'] = df_Coordinate_Identifiers['COORDINATE'].str.split(".").str[3]


df_Coordinate_Identifiers['NOC_GroupingID'] = df_Coordinate_Identifiers['NOC_GroupingID'].dropna().astype(int)
df_Coordinate_Identifiers['Gender_MemberID'] = df_Coordinate_Identifiers['Gender_MemberID'].dropna().astype(int)

df_Coordinate_Identifiers

,COORDINATE,NOC_GroupingID,Gender_MemberID
0,1.2.38.1,38.0,1.0
1,4.4.32.1,32.0,1.0
2,11.6.42.1,42.0,1.0
3,3.2.32.1,32.0,1.0
4,10.6.36.1,36.0,1.0
...,...,...,...
3956,2.1.26.1,26.0,1.0
3957,10.4.34.1,34.0,1.0
3958,6.4.34.1,34.0,1.0
3959,2.5.55.1,55.0,1.0


In [42]:
df_Coordinate_Identifiers.dtypes

COORDINATE          object
NOC_GroupingID     float64
Gender_MemberID    float64
dtype: object

In [43]:
ID_Count_Check = {
    'NOC_GroupingID': None,
    'Gender_MemberID': None,
}

for col in ID_Count_Check.keys():
    ID_Count_Check[col] = df_Coordinate_Identifiers[col].nunique()

ID_Count_Check

{'NOC_GroupingID': 60, 'Gender_MemberID': 1}

In [ ]:
#Verfied that there are 60 grouping and only 1 gender member id => which is Total Men + Women

In [44]:
Coordinate_NOC_Map_Dict = df_Coordinate_Identifiers.set_index('COORDINATE')['NOC_GroupingID'].to_dict()

In [56]:
Coordinate_NOC_Map_Dict

{'1.2.38.1': 38.0,
 '4.4.32.1': 32.0,
 '11.6.42.1': 42.0,
 '3.2.32.1': 32.0,
 '10.6.36.1': 36.0,
 '3.6.31.1': 31.0,
 '9.3.39.1': 39.0,
 '7.2.43.1': 43.0,
 '6.4.26.1': 26.0,
 '6.1.57.1': 57.0,
 '6.5.39.1': 39.0,
 '11.2.37.1': 37.0,
 '1.2.60.1': 60.0,
 '9.1.39.1': 39.0,
 '1.4.12.1': 12.0,
 '3.2.26.1': 26.0,
 '3.6.27.1': 27.0,
 '3.2.27.1': 27.0,
 '4.5.13.1': 13.0,
 '3.4.30.1': 30.0,
 '10.2.46.1': 46.0,
 '4.1.25.1': 25.0,
 '4.1.41.1': 41.0,
 '9.2.6.1': 6.0,
 '11.6.25.1': 25.0,
 '7.2.10.1': 10.0,
 '1.4.28.1': 28.0,
 '5.1.54.1': 54.0,
 '1.2.48.1': 48.0,
 '6.2.34.1': 34.0,
 '4.5.59.1': 59.0,
 '9.3.42.1': 42.0,
 '9.6.47.1': 47.0,
 '5.3.38.1': 38.0,
 '3.1.7.1': 7.0,
 '2.3.2.1': 2.0,
 '11.5.22.1': 22.0,
 '7.2.24.1': 24.0,
 '8.3.51.1': 51.0,
 '5.4.12.1': 12.0,
 '11.3.35.1': 35.0,
 '1.4.40.1': 40.0,
 '6.6.7.1': 7.0,
 '7.3.19.1': 19.0,
 '6.5.45.1': 45.0,
 '7.2.49.1': 49.0,
 '8.4.48.1': 48.0,
 '11.5.53.1': 53.0,
 '4.3.29.1': 29.0,
 '1.5.25.1': 25.0,
 '10.6.8.1': 8.0,
 '1.4.22.1': 22.0,
 '1.5.5.1': 5

In [48]:
df_Province_Identifiers = pd.read_csv('processed_data/Tbl_Provinces.csv')
df_NOC_Grouping_Identifiers = pd.read_csv('processed_data/Tbl_NOC_TS_Groupings.csv')

In [46]:
Province_DGUID_To_Province_ID_Dict = df_Province_Identifiers.set_index('Province_DGUID')['ProvinceID'].to_dict()
Province_DGUID_To_Province_ID_Dict 

{'2021A000259': 59,
 '2021A000224': 24,
 '2021A000211': 11,
 '2021A000247': 47,
 '2021A000246': 46,
 '2021A000213': 13,
 '2021A000210': 10,
 '2021A000260': 60,
 '2021A000261': 61,
 '2021A000248': 48,
 '2021A000212': 12,
 '2021A000262': 62,
 '2021A000235': 35}

In [70]:
ProvinceID_to_Province_Map = df_Province_Identifiers.set_index('ProvinceID')['Province'].to_dict()
ProvinceID_to_Province_Map

{59: 'British Columbia',
 24: 'Quebec',
 11: 'Prince Edward Island',
 47: 'Saskatchewan',
 46: 'Manitoba',
 13: 'New Brunswick',
 10: 'Newfoundland and Labrador',
 60: 'Yukon',
 61: 'Northwest Territories',
 48: 'Alberta',
 12: 'Nova Scotia',
 62: 'Nunavut',
 35: 'Ontario'}

In [50]:
NOC_GroupingID_NOC_Grouping_Name_Dict = df_NOC_Grouping_Identifiers.set_index('NOC_GroupingID')['NOC_Grouping'].to_dict()
NOC_GroupingID_NOC_Grouping_Name_Dict

{1: 'Total, all occupations',
 2: 'Management occupations',
 3: 'Legislative and senior management occupations',
 4: 'Specialized middle management occupations',
 5: 'Middle management occupations in retail and wholesale trade and customer services',
 6: 'Middle management occupations in trades, transportation, production and utilities',
 7: 'Business, finance and administration occupations, except management',
 8: 'Professional occupations in finance and business',
 9: 'Professional occupations in finance',
 10: 'Professional occupations in business',
 11: 'Administrative and financial supervisors and specialized administrative occupations',
 12: 'Administrative occupations and transportation logistics occupations',
 13: 'Administrative and financial support and supply chain logistics occupations',
 14: 'Natural and applied sciences and related occupations, except management',
 15: 'Professional occupations in natural and applied sciences',
 16: 'Professional occupations in natural sc

Processing TS Data 

In [54]:
df_TS_Data_Reduced = dp_Raw_Data.filter(
    (pl.col('Labour force characteristics').is_in(['Labour force',
    'Employment',
    'Part-time employment',
    'Full-time employment',
    'Unemployment',
    'Unemployment rate']))
    &
    #Not interested in unclassified occupations because they offer no value to the users
    (pl.col('National Occupational Classification (NOC)') != 'Unclassified occupations')
).select(['COORDINATE','REF_DATE','DGUID','Labour force characteristics','VALUE']).collect().to_pandas()




In [55]:
df_TS_Data_Reduced.head(10)

,COORDINATE,REF_DATE,DGUID,Labour force characteristics,VALUE
0,1.1.1.1,2000-01,2021A000011124,Labour force,15385.9
1,1.1.1.1,2000-02,2021A000011124,Labour force,15496.5
2,1.1.1.1,2000-03,2021A000011124,Labour force,15548.4
3,1.1.1.1,2000-04,2021A000011124,Labour force,15604.1
4,1.1.1.1,2000-05,2021A000011124,Labour force,15949.6
5,1.1.1.1,2000-06,2021A000011124,Labour force,16075.2
6,1.1.1.1,2000-07,2021A000011124,Labour force,16256.8
7,1.1.1.1,2000-08,2021A000011124,Labour force,16275.5
8,1.1.1.1,2000-09,2021A000011124,Labour force,15888.5
9,1.1.1.1,2000-10,2021A000011124,Labour force,15910.5


In [57]:
#We Start by Merging Province information and NOC Grouping Information, and drop nulls

df_TS_Data_Reduced['ProvinceID'] = df_TS_Data_Reduced['DGUID'].map(Province_DGUID_To_Province_ID_Dict)
df_TS_Data_Reduced['NOC_GroupingID'] = df_TS_Data_Reduced['COORDINATE'].map(Coordinate_NOC_Map_Dict)

In [61]:
df_TS_Data_Reduced = df_TS_Data_Reduced.dropna(subset=['ProvinceID','NOC_GroupingID'])

In [62]:
df_TS_Data_Reduced.head(10)

,COORDINATE,REF_DATE,DGUID,Labour force characteristics,VALUE,ProvinceID,NOC_GroupingID
109386,2.1.1.1,2000-01,2021A000210,Labour force,225.2,10.0,1.0
109387,2.1.1.1,2000-02,2021A000210,Labour force,226.3,10.0,1.0
109388,2.1.1.1,2000-03,2021A000210,Labour force,229.4,10.0,1.0
109389,2.1.1.1,2000-04,2021A000210,Labour force,230.5,10.0,1.0
109390,2.1.1.1,2000-05,2021A000210,Labour force,243.2,10.0,1.0
109391,2.1.1.1,2000-06,2021A000210,Labour force,246.5,10.0,1.0
109392,2.1.1.1,2000-07,2021A000210,Labour force,257.1,10.0,1.0
109393,2.1.1.1,2000-08,2021A000210,Labour force,248.1,10.0,1.0
109394,2.1.1.1,2000-09,2021A000210,Labour force,242.8,10.0,1.0
109395,2.1.1.1,2000-10,2021A000210,Labour force,238.8,10.0,1.0


In [64]:
#Now we pivot the table so that the Labor Characteristics are the Columns 
#and we shorten the table substantially 
df_TS_Pivot = df_TS_Data_Reduced.pivot_table(
    index=['ProvinceID','DGUID','NOC_GroupingID','REF_DATE'],
    columns='Labour force characteristics',
    values='VALUE'

).reset_index()

In [68]:
df_TS_Pivot.dtypes

Labour force characteristics
ProvinceID              float64
DGUID                    object
NOC_GroupingID          float64
REF_DATE                 object
Employment              float64
Full-time employment    float64
Labour force            float64
Part-time employment    float64
Unemployment            float64
Unemployment rate       float64
dtype: object

In [ ]:
print(df_TS_Pivot.head())

Labour force characteristics  ProvinceID        DGUID  NOC_GroupingID  \
0                                   10.0  2021A000210             1.0   
1                                   10.0  2021A000210             1.0   
2                                   10.0  2021A000210             1.0   
3                                   10.0  2021A000210             1.0   
4                                   10.0  2021A000210             1.0   

Labour force characteristics REF_DATE  Employment  Full-time employment  \
0                             2000-01       184.2                 151.6   
1                             2000-02       184.4                 152.1   
2                             2000-03       185.7                 152.5   
3                             2000-04       187.7                 156.7   
4                             2000-05       200.1                 170.9   

Labour force characteristics  Labour force  Part-time employment  \
0                                    225.2

In [71]:
df_TS_Pivot['NOC_Grouping'] = df_TS_Pivot['NOC_GroupingID'].map(NOC_GroupingID_NOC_Grouping_Name_Dict)
df_TS_Pivot['Province'] = df_TS_Pivot['ProvinceID'].map(ProvinceID_to_Province_Map)

In [73]:
df_TS_Pivot = df_TS_Pivot.rename(columns={'REF_DATE':'datestamp'})
df_TS_Pivot['datestamp'] = pd.to_datetime(df_TS_Pivot['datestamp'])

In [74]:
df_TS_Pivot.dtypes

Labour force characteristics
ProvinceID                     float64
DGUID                           object
NOC_GroupingID                 float64
datestamp               datetime64[ns]
Employment                     float64
Full-time employment           float64
Labour force                   float64
Part-time employment           float64
Unemployment                   float64
Unemployment rate              float64
NOC_Grouping                    object
Province                        object
dtype: object

In [ ]:
#Going to add 4month bollinger bands for front-end analysis opportunities 
#Bollinger bands are usually used in the stock market to asses the volatility of an asset 
#We can use to to asses the volatility of Employment

In [76]:
df_TS_Pivot = df_TS_Pivot.sort_values(by=['ProvinceID','NOC_GroupingID'], ascending=True)
df_TS_Pivot.head()

Labour force characteristics,ProvinceID,DGUID,NOC_GroupingID,datestamp,Employment,Full-time employment,Labour force,Part-time employment,Unemployment,Unemployment rate,NOC_Grouping,Province
0,10.0,2021A000210,1.0,2000-01-01,184.2,151.6,225.2,32.6,40.9,18.2,"Total, all occupations",Newfoundland and Labrador
1,10.0,2021A000210,1.0,2000-02-01,184.4,152.1,226.3,32.2,42.0,18.6,"Total, all occupations",Newfoundland and Labrador
2,10.0,2021A000210,1.0,2000-03-01,185.7,152.5,229.4,33.1,43.8,19.1,"Total, all occupations",Newfoundland and Labrador
3,10.0,2021A000210,1.0,2000-04-01,187.7,156.7,230.5,31.0,42.8,18.6,"Total, all occupations",Newfoundland and Labrador
4,10.0,2021A000210,1.0,2000-05-01,200.1,170.9,243.2,29.3,43.1,17.7,"Total, all occupations",Newfoundland and Labrador


In [77]:
df_TS_Pivot['Employment Moving Average'] = df_TS_Pivot.groupby(['ProvinceID','NOC_GroupingID'])['Full-time employment'].transform(lambda x: round(x.rolling(window=4, min_periods=1).mean(), 3))
df_TS_Pivot['Employment Moving SD'] = df_TS_Pivot.groupby(['ProvinceID','NOC_GroupingID'])['Full-time employment'].transform(lambda x: round(x.rolling(window=4, min_periods=1).std(), 3))

In [78]:
#We can create bands of volatility
#upper band =  Employment Moving Average + (2 * Employment Moving SD) 
#lower band = Employment Moving Average - (2 * Employment Moving SD)

df_TS_Pivot['Employment Upper Band'] = round(df_TS_Pivot['Employment Moving Average'] + (2 * df_TS_Pivot['Employment Moving SD']))
df_TS_Pivot['Employment Lower Band'] = round(df_TS_Pivot['Employment Moving Average'] - (2 * df_TS_Pivot['Employment Moving SD']))



In [87]:
df_TS_Pivot = df_TS_Pivot.reset_index(drop=True)

In [89]:
df_TS_Pivot

Labour force characteristics,ProvinceID,DGUID,NOC_GroupingID,datestamp,Employment,Full-time employment,Labour force,Part-time employment,Unemployment,Unemployment rate,NOC_Grouping,Province,Employment Moving Average,Employment Moving SD,Employment Upper Band,Employment Lower Band,EntryID
0,10.0,2021A000210,1.0,2000-01-01,184.2,151.6,225.2,32.6,40.9,18.2,"Total, all occupations",Newfoundland and Labrador,151.600,NaN,NaN,NaN,2
1,10.0,2021A000210,1.0,2000-02-01,184.4,152.1,226.3,32.2,42.0,18.6,"Total, all occupations",Newfoundland and Labrador,151.850,0.354,153.0,151.0,2
2,10.0,2021A000210,1.0,2000-03-01,185.7,152.5,229.4,33.1,43.8,19.1,"Total, all occupations",Newfoundland and Labrador,152.067,0.451,153.0,151.0,2
3,10.0,2021A000210,1.0,2000-04-01,187.7,156.7,230.5,31.0,42.8,18.6,"Total, all occupations",Newfoundland and Labrador,153.225,2.346,158.0,149.0,2
4,10.0,2021A000210,1.0,2000-05-01,200.1,170.9,243.2,29.3,43.1,17.7,"Total, all occupations",Newfoundland and Labrador,158.050,8.816,176.0,140.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181558,59.0,2021A000259,59.0,2025-05-01,19.3,18.3,20.8,NaN,1.6,7.7,"Labourers in processing, manufacturing and uti...",British Columbia,17.850,0.520,19.0,17.0,2
181559,59.0,2021A000259,59.0,2025-06-01,18.4,17.0,19.2,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,17.825,0.568,19.0,17.0,2
181560,59.0,2021A000259,59.0,2025-07-01,16.5,15.6,16.5,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,17.225,1.218,20.0,15.0,2
181561,59.0,2021A000259,59.0,2025-08-01,13.4,12.6,13.4,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,15.875,2.446,21.0,11.0,2


In [90]:
#To store in our database, an unique id per row is required 
df_TS_Pivot['EntryID'] = df_TS_Pivot.index + 1

In [91]:
df_TS_Pivot

Labour force characteristics,ProvinceID,DGUID,NOC_GroupingID,datestamp,Employment,Full-time employment,Labour force,Part-time employment,Unemployment,Unemployment rate,NOC_Grouping,Province,Employment Moving Average,Employment Moving SD,Employment Upper Band,Employment Lower Band,EntryID
0,10.0,2021A000210,1.0,2000-01-01,184.2,151.6,225.2,32.6,40.9,18.2,"Total, all occupations",Newfoundland and Labrador,151.600,NaN,NaN,NaN,1
1,10.0,2021A000210,1.0,2000-02-01,184.4,152.1,226.3,32.2,42.0,18.6,"Total, all occupations",Newfoundland and Labrador,151.850,0.354,153.0,151.0,2
2,10.0,2021A000210,1.0,2000-03-01,185.7,152.5,229.4,33.1,43.8,19.1,"Total, all occupations",Newfoundland and Labrador,152.067,0.451,153.0,151.0,3
3,10.0,2021A000210,1.0,2000-04-01,187.7,156.7,230.5,31.0,42.8,18.6,"Total, all occupations",Newfoundland and Labrador,153.225,2.346,158.0,149.0,4
4,10.0,2021A000210,1.0,2000-05-01,200.1,170.9,243.2,29.3,43.1,17.7,"Total, all occupations",Newfoundland and Labrador,158.050,8.816,176.0,140.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181558,59.0,2021A000259,59.0,2025-05-01,19.3,18.3,20.8,NaN,1.6,7.7,"Labourers in processing, manufacturing and uti...",British Columbia,17.850,0.520,19.0,17.0,181559
181559,59.0,2021A000259,59.0,2025-06-01,18.4,17.0,19.2,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,17.825,0.568,19.0,17.0,181560
181560,59.0,2021A000259,59.0,2025-07-01,16.5,15.6,16.5,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,17.225,1.218,20.0,15.0,181561
181561,59.0,2021A000259,59.0,2025-08-01,13.4,12.6,13.4,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,15.875,2.446,21.0,11.0,181562


In [92]:
#Counting Nulls for Future Reference 
df_TS_Pivot.isna().sum()

Labour force characteristics
ProvinceID                       0
DGUID                            0
NOC_GroupingID                   0
datestamp                        0
Employment                     177
Full-time employment          1453
Labour force                     0
Part-time employment         42113
Unemployment                 90720
Unemployment rate            90720
NOC_Grouping                     0
Province                         0
Employment Moving Average      358
Employment Moving SD          1421
Employment Upper Band         1421
Employment Lower Band         1421
EntryID                          0
dtype: int64

In [93]:
df_TS_Pivot['EntryID'].nunique()

181563

In [94]:
df_TS_Pivot.to_parquet('processed_data/Tbl_Provincial_NOC_Labor_Statistics.parquet',index=False)

In [1]:
import pandas as pd 

In [2]:
df = pd.read_parquet('processed_data/Tbl_Provincial_NOC_Labor_Statistics.parquet')

df

,ProvinceID,DGUID,NOC_GroupingID,datestamp,Employment,Full-time employment,Labour force,Part-time employment,Unemployment,Unemployment rate,NOC_Grouping,Province,Employment Moving Average,Employment Moving SD,Employment Upper Band,Employment Lower Band,EntryID
0,10.0,2021A000210,1.0,2000-01-01,184.2,151.6,225.2,32.6,40.9,18.2,"Total, all occupations",Newfoundland and Labrador,151.600,NaN,NaN,NaN,1
1,10.0,2021A000210,1.0,2000-02-01,184.4,152.1,226.3,32.2,42.0,18.6,"Total, all occupations",Newfoundland and Labrador,151.850,0.354,153.0,151.0,2
2,10.0,2021A000210,1.0,2000-03-01,185.7,152.5,229.4,33.1,43.8,19.1,"Total, all occupations",Newfoundland and Labrador,152.067,0.451,153.0,151.0,3
3,10.0,2021A000210,1.0,2000-04-01,187.7,156.7,230.5,31.0,42.8,18.6,"Total, all occupations",Newfoundland and Labrador,153.225,2.346,158.0,149.0,4
4,10.0,2021A000210,1.0,2000-05-01,200.1,170.9,243.2,29.3,43.1,17.7,"Total, all occupations",Newfoundland and Labrador,158.050,8.816,176.0,140.0,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181558,59.0,2021A000259,59.0,2025-05-01,19.3,18.3,20.8,NaN,1.6,7.7,"Labourers in processing, manufacturing and uti...",British Columbia,17.850,0.520,19.0,17.0,181559
181559,59.0,2021A000259,59.0,2025-06-01,18.4,17.0,19.2,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,17.825,0.568,19.0,17.0,181560
181560,59.0,2021A000259,59.0,2025-07-01,16.5,15.6,16.5,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,17.225,1.218,20.0,15.0,181561
181561,59.0,2021A000259,59.0,2025-08-01,13.4,12.6,13.4,NaN,NaN,NaN,"Labourers in processing, manufacturing and uti...",British Columbia,15.875,2.446,21.0,11.0,181562


In [3]:
df_Provinces = df[['ProvinceID','DGUID','Province']].drop_duplicates()
df_Provinces

,ProvinceID,DGUID,Province
0,10.0,2021A000210,Newfoundland and Labrador
18001,11.0,2021A000211,Prince Edward Island
35805,12.0,2021A000212,Nova Scotia
54026,13.0,2021A000213,New Brunswick
72183,24.0,2021A000224,Quebec
90414,35.0,2021A000235,Ontario
108645,46.0,2021A000246,Manitoba
126875,47.0,2021A000247,Saskatchewan
145102,48.0,2021A000248,Alberta
163332,59.0,2021A000259,British Columbia
